# Step 1 - ADE20K 시맨틱 세그멘테이션 (시야 차단 구조물 분리)

**목표**: Front view에서 자율주행 시야를 차단하는 **정적 구조물**(건물, 벽, 울타리, 가로수, 기둥 등)을 픽셀 단위로 분리한다. 제안서 1단계의 방음벽·옹벽·가로수·교각은 COCO에 없으므로 도시 구조물 클래스를 가진 **ADE20K(150-class) SegFormer**로 교체한다.

추가로 **지면/하늘 클래스**(road/sidewalk/earth/sky 등)도 별도 마스크(`ground_mask`)로 저장한다. 03 BEV 투영 시 이 픽셀을 제외하면 도로 바닥이 BEV에 찍혀 L_vis가 과대 추정되는 문제를 막을 수 있다.

**입력**: `output/00_front/{stem}_front.jpg` (Step 0 Front view)

**출력**: `output/01_seg/{stem}_masks.npz` (`masks`=구조물, `ground_mask`=지면/하늘) + `{stem}_meta.json` + `{stem}_seg_vis.jpg`

## 0. 패키지 설치 (가상환경 kernel 기준, 처음 1회)

GPU(RTX 3070 Ti, CUDA 12.x)를 쓰도록 CUDA 빌드 torch를 설치한다.

In [1]:
# CUDA 12.4 torch (GPU) - 이미 CUDA 빌드가 깔려 있으면 건너뛰어도 됨
# !pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124
# !pip install transformers accelerate pillow opencv-python-headless matplotlib numpy

## 1. 라이브러리 로드 및 경로 설정

In [2]:
import json
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import torch

# 경로 설정
IMG_DIR = Path("output/00_front")   # Step 0 Front view
OUT_DIR = Path("output/01_seg")
OUT_DIR.mkdir(parents=True, exist_ok=True)

IMG_PATHS = sorted(IMG_DIR.glob("*_front.jpg"))
print(f"Front view {len(IMG_PATHS)}장 발견")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"사용 디바이스: {device}")
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Front view 1137장 발견
  cut_point_1000_pano_4-zgBUPlEx-LFtwRNtIEiQ_front.jpg
  cut_point_1001_pano_IQ4H3AfYxt5vQoSdRa1aUw_front.jpg
  cut_point_1005_pano_sSNXY4Ndp-CnY4O7DZcjPg_front.jpg
  cut_point_1006_pano_Rr8L4nop0nWbNMkf-dzYxg_front.jpg
  cut_point_1007_pano_JCJoTj97-psyPNnNt73bbQ_front.jpg
  cut_point_1008_pano_aHMpHsJEwWH-FjvMTMOsUA_front.jpg
  cut_point_1009_pano_XeD5q-0U1WD2I4FXfRjoAA_front.jpg
  cut_point_100_pano_3h8UuIgEVhDpJILYS2UlFQ_front.jpg
  cut_point_1010_pano_oaDIYeTodpyj-peusZKC0g_front.jpg
  cut_point_1014_pano_4IdLIexntFaB_qQOOWNX0g_front.jpg
  cut_point_1016_pano_KD_eTNUuea9JamXyKe6sCA_front.jpg
  cut_point_101_pano_-leTuFvUyJTvoRgnijTOqQ_front.jpg
  cut_point_1026_pano_tnNrfDc7Hb_L2h825m2A9Q_front.jpg
  cut_point_1028_pano_daqdTzaIMKsuUxJRKXndDA_front.jpg
  cut_point_1029_pano_1BliXBrc9XXRTFqBrQdxUQ_front.jpg
  cut_point_102_pano_dSZMzdPqmuvZLLezCn5soA_front.jpg
  cut_point_1030_pano_ilWx1kc34kQS5D-3sUt2RQ_front.jpg
  cut_point_1033_pano_u4ix2_FhxIoldyEnRlRDOQ_fro

## 2. ADE20K SegFormer 모델 로드

`nvidia/segformer-b5-finetuned-ade-640-640` (mIoU ~51.8, VRAM ~4GB). 처음 실행 시 HF에서 자동 다운로드.

In [3]:
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor

MODEL_ID = "nvidia/segformer-b5-finetuned-ade-640-640"
print(f"ADE20K SegFormer 로드 중: {MODEL_ID}")

processor = SegformerImageProcessor.from_pretrained(MODEL_ID)
model = SegformerForSemanticSegmentation.from_pretrained(MODEL_ID).to(device).eval()

id2label = model.config.id2label   # {idx: 'wall', ...}
print("모델 로드 완료. 클래스 수:", len(id2label))

c:\Users\Jihyun\Desktop\git\ITS\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ADE20K SegFormer 로드 중: nvidia/segformer-b5-finetuned-ade-640-640


Loading weights: 100%|██████████| 1172/1172 [00:00<00:00, 56056.06it/s]


모델 로드 완료. 클래스 수: 150


## 3. '시야 차단 구조물' 클래스 화이트리스트

ADE20K 150클래스 중 자율주행 시야를 막는 **정적 수직 구조물**을 이름 기반으로 선택한다. (인덱스 하드코딩 대신 모델의 `id2label`에서 이름 매칭 → 모델이 바뀌어도 안전)

Road/하늘/보도/차량 등 시야를 막지 **않는** 클래스는 제외(차량은 동적이라 제안서 4단계 V_heavy 영역).

In [4]:
# 시야 차단 구조물로 간주할 클래스 이름 키워드
BLOCKER_KEYWORDS = [
    # 건물·벽 계열
    "building", "wall", "house", "skyscraper", "hovel", "tower",
    # 차폐물·구조재
    "fence", "hedge", "railing", "column", "pillar", "bannister",
    "canopy", "grandstand", "bulletin board",
    # 식생
    "tree", "plant", "palm", "flower",
    # 기둥·표지물
    "pole", "awning", "signboard", "booth", "bridge", "streetlight",
    "sculpture", "flag",
    # 지형·자연 차단체 (산지·암벽 Road)
    "mountain", "hill", "rock",
    # 임시·행사 구조물
    "tent",
    # 계단·고가 구조물
    "stairs", "stairway",
    # 스크린형 차단체
    "screen door",
]

# 지면/하늘 등 '시야를 막지 않는 평면' 클래스 키워드 (03 BEV 투영에서 제외 대상)
GROUND_KEYWORDS = [
    "road", "sidewalk", "floor", "earth", "field", "grass",
    "path", "runway", "dirt track", "land", "sky",
]

# id2label에서 키워드가 포함된 클래스만 선택
BLOCKER_IDS = {}
for idx, name in id2label.items():
    idx = int(idx)
    if any(kw in name.lower() for kw in BLOCKER_KEYWORDS):
        BLOCKER_IDS[idx] = name

GROUND_IDS = {}
for idx, name in id2label.items():
    idx = int(idx)
    if any(kw in name.lower() for kw in GROUND_KEYWORDS):
        GROUND_IDS[idx] = name

print(f"시야 차단 구조물 클래스 {len(BLOCKER_IDS)}개:")
for i, n in sorted(BLOCKER_IDS.items()):
    print(f"  [{i:3d}] {n}")

print(f"\n지면/하늘 클래스 {len(GROUND_IDS)}개 (BEV 제외 대상):")
for i, n in sorted(GROUND_IDS.items()):
    print(f"  [{i:3d}] {n}")

# 시각화용 색상 (클래스 id -> BGR), 결정적 난수
rng = np.random.default_rng(42)
CLASS_COLORS = {i: tuple(int(c) for c in rng.integers(40, 230, 3)) for i in BLOCKER_IDS}

시야 차단 구조물 클래스 32개:
  [  0] wall
  [  1] building
  [  4] tree
  [ 16] mountain
  [ 17] plant
  [ 25] house
  [ 32] fence
  [ 34] rock
  [ 38] railing
  [ 42] column
  [ 43] signboard
  [ 48] skyscraper
  [ 51] grandstand
  [ 53] stairs
  [ 58] screen door
  [ 59] stairway
  [ 61] bridge
  [ 66] flower
  [ 68] hill
  [ 72] palm
  [ 79] hovel
  [ 84] tower
  [ 86] awning
  [ 87] streetlight
  [ 88] booth
  [ 93] pole
  [ 95] bannister
  [106] canopy
  [114] tent
  [132] sculpture
  [144] bulletin board
  [149] flag

지면/하늘 클래스 13개 (BEV 제외 대상):
  [  2] sky
  [  3] floor
  [  6] road
  [  9] grass
  [ 11] sidewalk
  [ 13] earth
  [ 29] field
  [ 48] skyscraper
  [ 52] path
  [ 54] runway
  [ 73] kitchen island
  [ 91] dirt track
  [ 94] land


## 4. 세그멘테이션 함수

이미지 1장 -> 전체 픽셀 클래스 라벨맵 -> 구조물 클래스만 클래스별 마스크로 추출.

In [5]:
@torch.no_grad()
def segment_image(img_bgr):
    # BGR 이미지 -> (검출 결과 리스트, 전체 레이블맵, 지면/하늘 마스크)
    h, w = img_bgr.shape[:2]
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    inputs = processor(images=Image.fromarray(img_rgb), return_tensors="pt").to(device)

    logits = model(**inputs).logits   # (1, C, h/4, w/4)
    up = torch.nn.functional.interpolate(
        logits, size=(h, w), mode="bilinear", align_corners=False)
    label_map = up.argmax(dim=1)[0].cpu().numpy().astype(np.int32)  # (H, W)

    # GPU 텐서 즉시 해제
    del inputs, logits, up
    if device == "cuda":
        torch.cuda.empty_cache()

    detections = []
    for cls_id, cls_name in BLOCKER_IDS.items():
        mask = label_map == cls_id
        area = int(mask.sum())
        if area < 200:   # 너무 작은 영역은 노이즈로 무시
            continue
        detections.append({
            "class_id": cls_id,
            "class_name": cls_name,
            "mask": mask,
            "area_px": area,
        })
    detections.sort(key=lambda d: -d["area_px"])   # 넓은 순 정렬

    # 지면/하늘 마스크 (03 BEV 투영에서 제외용). 별도 클래스 집합으로.
    ground_mask = np.isin(label_map, list(GROUND_IDS.keys()))
    return detections, label_map, ground_mask


print("세그멘테이션 함수 정의 완료")


세그멘테이션 함수 정의 완료


## 5. 시각화 함수

구조물 마스크를 반투명 오버레이로 표시. (이전 버전의 in-place 버그 수정: 단일 uint8 배열에 그림)

In [6]:
def visualize_segmentation(img_bgr, detections, title=""):
    overlay = img_bgr.astype(np.float32)
    for det in detections:
        color = CLASS_COLORS.get(det["class_id"], (128, 128, 128))
        mask = det["mask"]
        for c, val in enumerate(color):
            overlay[:, :, c][mask] = overlay[:, :, c][mask] * 0.45 + val * 0.55

    vis = np.clip(overlay, 0, 255).astype(np.uint8)   # 한 번만 변환

    seen = {}
    for det in detections:
        cid = det["class_id"]
        if cid not in seen:
            bgr = CLASS_COLORS.get(cid, (128, 128, 128))
            seen[cid] = mpatches.Patch(color=[bgr[2]/255, bgr[1]/255, bgr[0]/255],
                                       label=det["class_name"])
    patches = list(seen.values())

    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.set_title(title, fontsize=12)
    ax.axis("off")
    if patches:
        ax.legend(handles=patches, loc="upper right", fontsize=8)
    plt.tight_layout()
    return fig, vis


print("시각화 함수 정의 완료")

시각화 함수 정의 완료


## 6. 전체 이미지에 파이프라인 실행

In [7]:
from tqdm.auto import tqdm

all_results = {}

# 이미 처리된 파일 로드 (이어서 실행 지원)
done = set()
for p in OUT_DIR.glob("*_masks.npz"):
    stem = p.stem.replace("_masks", "")
    meta_path = OUT_DIR / f"{stem}_meta.json"
    if meta_path.exists():
        all_results[stem] = json.load(open(meta_path, encoding="utf-8"))["detections"]
        done.add(stem)

remaining = [p for p in IMG_PATHS if p.stem not in done]
print(f"이미 완료: {len(done)}장 / 전체: {len(IMG_PATHS)}장 → 남은 작업: {len(remaining)}장")

for img_path in tqdm(remaining, desc="세그멘테이션"):
    stem = img_path.stem
    img_bgr = cv2.imread(str(img_path))
    h, w = img_bgr.shape[:2]

    detections, label_map, ground_mask = segment_image(img_bgr)

    # 시각화 저장
    fig, vis_img = visualize_segmentation(
        img_bgr, detections, title=f"{img_path.name}  |  ADE20K blockers")
    cv2.imwrite(str(OUT_DIR / f"{stem}_seg_vis.jpg"), vis_img)
    plt.close(fig)
    del vis_img, fig

    # 마스크 저장 (.npz)
    if detections:
        masks_array = np.stack([d["mask"] for d in detections], axis=0)  # (N,H,W)
        meta = [{"class_id": d["class_id"], "class_name": d["class_name"],
                 "area_px": d["area_px"]} for d in detections]
    else:
        masks_array = np.zeros((0, h, w), dtype=bool)
        meta = []

    np.savez_compressed(str(OUT_DIR / f"{stem}_masks.npz"),
                        masks=masks_array, ground_mask=ground_mask)
    with open(OUT_DIR / f"{stem}_meta.json", "w", encoding="utf-8") as f:
        json.dump({"image": img_path.name, "img_hw": [h, w], "detections": meta,
                   "ground_px": int(ground_mask.sum())},
                  f, ensure_ascii=False, indent=2)

    # mask 배열은 저장 후 즉시 해제 (all_results에는 meta만 보관)
    all_results[stem] = meta
    del detections, masks_array, ground_mask, label_map, img_bgr

print("=== 완료 ===")


세그멘테이션: 100%|██████████| 1137/1137 [01:57<00:00,  9.71it/s]

=== 완료 ===


## 7. 요약

In [8]:
total_detections = sum(len(v) for v in all_results.values())
print(f"=== 세그멘테이션 요약 ({len(all_results)}장) ===")
print(f"  총 검출 객체 수: {total_detections}개")
print(f"  저장 위치: {OUT_DIR}")
print("다음 단계: 02_depth_estimation.ipynb")

=== 세그멘테이션 결과 요약 ===
cut_point_1000_pano_4-zgBUPlEx-LFtwRNtIEiQ_fr
  구조물 클래스: ['building', 'pole']
  총 마스크 픽셀: 37,125
cut_point_1001_pano_IQ4H3AfYxt5vQoSdRa1aUw_fr
  구조물 클래스: ['building', 'tree']
  총 마스크 픽셀: 27,868
cut_point_1005_pano_sSNXY4Ndp-CnY4O7DZcjPg_fr
  구조물 클래스: ['building', 'tree']
  총 마스크 픽셀: 26,503
cut_point_1006_pano_Rr8L4nop0nWbNMkf-dzYxg_fr
  구조물 클래스: ['building', 'plant', 'tree', 'wall']
  총 마스크 픽셀: 29,153
cut_point_1007_pano_JCJoTj97-psyPNnNt73bbQ_fr
  구조물 클래스: ['building', 'fence']
  총 마스크 픽셀: 35,740
cut_point_1008_pano_aHMpHsJEwWH-FjvMTMOsUA_fr
  구조물 클래스: ['building', 'tree', 'awning', 'plant']
  총 마스크 픽셀: 30,279
cut_point_1009_pano_XeD5q-0U1WD2I4FXfRjoAA_fr
  구조물 클래스: ['building', 'tree', 'stairs', 'plant']
  총 마스크 픽셀: 32,499
cut_point_100_pano_3h8UuIgEVhDpJILYS2UlFQ_fro
  구조물 클래스: ['building', 'signboard', 'wall', 'tree']
  총 마스크 픽셀: 37,567
cut_point_1010_pano_oaDIYeTodpyj-peusZKC0g_fr
  구조물 클래스: ['building', 'column']
  총 마스크 픽셀: 35,051
cut_point_1014_pano_4IdLIex